<a href="https://colab.research.google.com/github/jaiswal09/AQi/blob/main/Group3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import numpy as np
import pandas as pd
import time
import json
import sys
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import xgboost as xgb

# ========================================== #
# SECTION 1: Exploratory Data Analysis (EDA) #
# ========================================== #

base_dir = '/content'
eda_dir = os.path.join(base_dir, 'eda_plots')
os.makedirs(eda_dir, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8
plt.rcParams['figure.dpi'] = 150

print("--- STARTING EDA ---")
train = pd.read_csv(os.path.join(base_dir, 'beijing_train.csv'))
test = pd.read_csv(os.path.join(base_dir, 'beijing_test.csv'))

train['datetime'] = pd.to_datetime(train[['year', 'month', 'day', 'hour']])
test['datetime'] = pd.to_datetime(test[['year', 'month', 'day', 'hour']])
train_clean = train.dropna(subset=['PM2.5']).copy()

# Plot 1
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(train_clean['PM2.5'], bins=100, kde=True, color='#2b5c8f', ax=axes[0])
axes[0].set_title('Distribution of PM2.5 Concentration (Raw)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('PM2.5 (µg/m³)', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].axvline(train_clean['PM2.5'].mean(), color='red', linestyle='--', label=f"Mean: {train_clean['PM2.5'].mean():.1f}")
axes[0].axvline(train_clean['PM2.5'].median(), color='green', linestyle='-', label=f"Median: {train_clean['PM2.5'].median():.1f}")
axes[0].axvline(35, color='orange', linestyle=':', label="WHO Annual Guideline (35 µg/m³)")
axes[0].legend(frameon=True)

sns.histplot(np.log1p(train_clean['PM2.5']), bins=80, kde=True, color='#1f77b4', ax=axes[1])
axes[1].set_title('Distribution of log(1 + PM2.5)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('log(1 + PM2.5)', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(eda_dir, '01_pm25_distribution.png'), bbox_inches='tight')
plt.close()

# Plot 2
fig, ax = plt.subplots(figsize=(11, 9))
num_cols = ['PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM']
corr = train_clean[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-0.6, vmax=1.0, mask=mask,
            linewidths=0.5, cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('Pearson Correlation Matrix (Air Pollutants & Weather)', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(os.path.join(eda_dir, '02_correlation_heatmap.png'), bbox_inches='tight')
plt.close()

# Plot 3
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
hb1 = axes[0].hexbin(train_clean['PM10'], train_clean['PM2.5'], gridsize=60, cmap='Blues', mincnt=1, bins='log')
axes[0].plot([0, 800], [0, 800], 'r--', label='1:1 Line (PM2.5 = PM10)')
axes[0].set_xlim(0, 800)
axes[0].set_ylim(0, 800)
axes[0].set_title('Relationship: PM2.5 vs PM10 (Log Count)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('PM10 (µg/m³)', fontsize=11)
axes[0].set_ylabel('PM2.5 (µg/m³)', fontsize=11)
axes[0].legend()
fig.colorbar(hb1, ax=axes[0], label='log10(count)')

hb2 = axes[1].hexbin(train_clean['CO'], train_clean['PM2.5'], gridsize=60, cmap='Purples', mincnt=1, bins='log')
axes[1].set_xlim(0, 8000)
axes[1].set_ylim(0, 800)
axes[1].set_title('Relationship: PM2.5 vs CO (Log Count)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('CO (µg/m³)', fontsize=11)
axes[1].set_ylabel('PM2.5 (µg/m³)', fontsize=11)
fig.colorbar(hb2, ax=axes[1], label='log10(count)')
plt.tight_layout()
plt.savefig(os.path.join(eda_dir, '03_pm25_vs_pm10_co.png'), bbox_inches='tight')
plt.close()

# Plot 4
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
monthly_avg = train_clean.groupby('month')['PM2.5'].mean().reset_index()
monthly_med = train_clean.groupby('month')['PM2.5'].median().reset_index()
axes[0].plot(monthly_avg['month'], monthly_avg['PM2.5'], marker='o', linewidth=2.5, color='#d95f02', label='Mean')
axes[0].plot(monthly_med['month'], monthly_med['PM2.5'], marker='s', linewidth=2, linestyle='--', color='#7570b3', label='Median')
axes[0].set_title('Seasonal Pattern: Average PM2.5 by Month', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Month', fontsize=11)
axes[0].set_ylabel('PM2.5 (µg/m³)', fontsize=11)
axes[0].set_xticks(range(1, 13))
axes[0].set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
axes[0].axvspan(11, 12, color='orange', alpha=0.15, label='Heating Season (Nov-Mar)')
axes[0].axvspan(1, 3.5, color='orange', alpha=0.15)
axes[0].legend()

hourly_avg = train_clean.groupby('hour')['PM2.5'].mean().reset_index()
hourly_med = train_clean.groupby('hour')['PM2.5'].median().reset_index()
axes[1].plot(hourly_avg['hour'], hourly_avg['PM2.5'], marker='o', linewidth=2.5, color='#1b9e77', label='Mean')
axes[1].plot(hourly_med['hour'], hourly_med['PM2.5'], marker='s', linewidth=2, linestyle='--', color='#e7298a', label='Median')
axes[1].set_title('Diurnal Pattern: Average PM2.5 by Hour of Day', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Hour of Day (0-23)', fontsize=11)
axes[1].set_ylabel('PM2.5 (µg/m³)', fontsize=11)
axes[1].set_xticks(range(0, 24, 2))
axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(eda_dir, '04_temporal_trends.png'), bbox_inches='tight')
plt.close()

# Plot 5
fig, ax = plt.subplots(figsize=(14, 6))
station_order = train_clean.groupby('station')['PM2.5'].median().sort_values().index
sns.boxplot(data=train_clean, x='station', y='PM2.5', order=station_order, hue='station', palette='Spectral',
            legend=False, showfliers=False, ax=ax)
ax.set_title('PM2.5 Distribution Across 12 Beijing Monitoring Stations', fontsize=14, fontweight='bold')
ax.set_xlabel('Station', fontsize=12)
ax.set_ylabel('PM2.5 (µg/m³)', fontsize=12)
ax.set_xticks(range(len(station_order)))
ax.set_xticklabels(station_order, rotation=30, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(eda_dir, '05_station_distribution.png'), bbox_inches='tight')
plt.close()

# Plot 6
wd_order = ['N', 'NNE', 'NE', 'ENE', 'E', 'ESE', 'SE', 'SSE', 'S', 'SSW', 'SW', 'WSW', 'W', 'WNW', 'NW', 'NNW']
train_clean_wd = train_clean.dropna(subset=['wd'])
wd_stats = train_clean_wd.groupby('wd').agg(
    mean_pm25=('PM2.5', 'mean'),
    median_pm25=('PM2.5', 'median'),
    mean_wspm=('WSPM', 'mean'),
    count=('PM2.5', 'count')
).reindex(wd_order)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
bars = axes[0].bar(wd_stats.index, wd_stats['mean_pm25'], color=plt.cm.YlOrRd(wd_stats['mean_pm25'] / wd_stats['mean_pm25'].max()))
axes[0].set_title('Mean PM2.5 Concentration by Wind Direction', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Wind Direction (16 Compass Points)', fontsize=11)
axes[0].set_ylabel('Mean PM2.5 (µg/m³)', fontsize=11)
axes[0].set_xticks(range(len(wd_order)))
axes[0].set_xticklabels(wd_stats.index, rotation=45)

sample_sub = train_clean.sample(min(40000, len(train_clean)), random_state=42)
axes[1].scatter(sample_sub['WSPM'], sample_sub['PM2.5'], alpha=0.15, color='#2b5c8f', s=10)
wspm_bins = pd.cut(train_clean['WSPM'], bins=np.linspace(0, 10, 21))
wspm_pm25_med = train_clean.groupby(wspm_bins, observed=False)['PM2.5'].median()
bin_centers = [interval.mid for interval in wspm_pm25_med.index]
axes[1].plot(bin_centers, wspm_pm25_med.values, color='red', linewidth=3, marker='o', label='Median PM2.5')
axes[1].set_title('PM2.5 vs Wind Speed (Dispersal Effect)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Wind Speed (m/s)', fontsize=11)
axes[1].set_ylabel('PM2.5 (µg/m³)', fontsize=11)
axes[1].set_xlim(0, 10)
axes[1].set_ylim(0, 500)
axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(eda_dir, '06_wind_effects.png'), bbox_inches='tight')
plt.close()

# Plot 7
fig, ax = plt.subplots(figsize=(12, 5))
yearly_stn = train_clean.groupby(['year', 'month'])['PM2.5'].mean().unstack(level=0)
for yr in [2013, 2014, 2015, 2016]:
    if yr in yearly_stn.columns:
        ax.plot(yearly_stn.index, yearly_stn[yr], marker='o', label=str(yr), linewidth=2)
ax.set_title('Monthly PM2.5 Trend Across Years (2013 - 2016)', fontsize=13, fontweight='bold')
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Mean PM2.5 (µg/m³)', fontsize=11)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
ax.legend(title='Year')
plt.tight_layout()
plt.savefig(os.path.join(eda_dir, '07_yearly_trends.png'), bbox_inches='tight')
plt.close()

# Plot 8
train_clean['pm_ratio'] = train_clean['PM2.5'] / (train_clean['PM10'] + 1e-5)
valid_ratio = train_clean[train_clean['pm_ratio'] <= 1.5]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(valid_ratio['pm_ratio'], bins=75, kde=True, color='#4575b4', ax=axes[0])
axes[0].set_title('PM2.5 to PM10 Ratio Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('PM2.5 / PM10 Ratio', fontsize=11)
axes[0].axvline(valid_ratio['pm_ratio'].median(), color='red', linestyle='--',
                label=f"Median Ratio: {valid_ratio['pm_ratio'].median():.2f}")
axes[0].legend()

train_clean['season'] = train_clean['month'].map({
    12: 'Winter', 1: 'Winter', 2: 'Winter',
    3: 'Spring', 4: 'Spring', 5: 'Spring',
    6: 'Summer', 7: 'Summer', 8: 'Summer',
    9: 'Autumn', 10: 'Autumn', 11: 'Autumn'
})
season_order = ['Winter', 'Spring', 'Summer', 'Autumn']
sns.boxplot(data=train_clean[train_clean['pm_ratio'] <= 1.5], x='season', y='pm_ratio',
            order=season_order, hue='season', palette='Set2', legend=False, showfliers=False, ax=axes[1])
axes[1].set_title('PM2.5 / PM10 Ratio by Season', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Season', fontsize=11)
axes[1].set_ylabel('PM2.5 / PM10 Ratio', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(eda_dir, '08_pm_ratio_seasonal.png'), bbox_inches='tight')
plt.close()

print("All 8 EDA figures successfully saved to eda_plots/!")

# ========================================================================= #
# SECTION 2: COMPREHENSIVE FEATURE ENGINEERING, BENCHMARK, & MODEL ALIGNMENT #
# ========================================================================= #

model_dir = os.path.join(base_dir, 'model_plots')
os.makedirs(model_dir, exist_ok=True)
print("\n=== STARTING COMPREHENSIVE FEATURE ENGINEERING & MODEL BENCHMARK ===")
start_time = time.time()

# Tag original order so row ordering is 100% strictly preserved
train = pd.read_csv(os.path.join(base_dir, 'beijing_train.csv'))
test = pd.read_csv(os.path.join(base_dir, 'beijing_test.csv'))

train['orig_order'] = np.arange(len(train))
test['orig_order'] = np.arange(len(test))
train['is_test'] = 0
test['is_test'] = 1

train['datetime'] = pd.to_datetime(train[['year', 'month', 'day', 'hour']])
test['datetime'] = pd.to_datetime(test[['year', 'month', 'day', 'hour']])

test['PM2.5'] = np.nan
combined = pd.concat([train, test], ignore_index=True)
combined = combined.sort_values(['station', 'datetime']).reset_index(drop=True)

# Wind Direction & Aerodynamics
wd_angles = {
    'N': 0, 'NNE': 22.5, 'NE': 45, 'ENE': 67.5,
    'E': 90, 'ESE': 112.5, 'SE': 135, 'SSE': 157.5,
    'S': 180, 'SSW': 202.5, 'SW': 225, 'WSW': 247.5,
    'W': 270, 'WNW': 292.5, 'NW': 315, 'NNW': 337.5
}
combined['wd_angle'] = combined['wd'].map(wd_angles)
combined['sin_wd'] = np.sin(np.radians(combined['wd_angle']))
combined['cos_wd'] = np.cos(np.radians(combined['wd_angle']))
combined['u_wind'] = -combined['WSPM'] * combined['sin_wd']
combined['v_wind'] = -combined['WSPM'] * combined['cos_wd']

# Atmospheric Thermodynamics & Moisture
def calc_rh(temp, dewp):
    a, b = 17.625, 243.04
    alpha = (a * temp) / (b + temp)
    beta = (a * dewp) / (b + dewp)
    rh = 100.0 * np.exp(beta - alpha)
    return np.clip(rh, 0, 100)

combined['RH'] = calc_rh(combined['TEMP'], combined['DEWP'])
combined['dew_depression'] = combined['TEMP'] - combined['DEWP']

es = 0.61078 * np.exp((17.27 * combined['TEMP']) / (combined['TEMP'] + 237.3))
ea = 0.61078 * np.exp((17.27 * combined['DEWP']) / (combined['DEWP'] + 237.3))
combined['VPD'] = np.maximum(0, es - ea)
combined['air_density'] = combined['PRES'] / (2.87 * (combined['TEMP'] + 273.15))

combined['dispersion_factor'] = combined['WSPM'] / (combined['dew_depression'] + 1.0)
combined['stagnation_flag'] = ((combined['WSPM'] < 1.5) & (combined['RH'] > 70)).astype(int)

# Temporal & Calendar
combined['hour_sin'] = np.sin(2 * np.pi * combined['hour'] / 24.0)
combined['hour_cos'] = np.cos(2 * np.pi * combined['hour'] / 24.0)
combined['month_sin'] = np.sin(2 * np.pi * combined['month'] / 12.0)
combined['month_cos'] = np.cos(2 * np.pi * combined['month'] / 12.0)
doy = combined['datetime'].dt.day_of_year
combined['doy_sin'] = np.sin(2 * np.pi * doy / 365.25)
combined['doy_cos'] = np.cos(2 * np.pi * doy / 365.25)
combined['dayofweek'] = combined['datetime'].dt.day_of_week
combined['is_weekend'] = (combined['dayofweek'] >= 5).astype(int)
combined['is_heating'] = ((combined['month'] >= 11) | (combined['month'] <= 3)).astype(int)

# Chemical Indices
combined['pm10_co_ratio'] = combined['PM10'] / (combined['CO'] + 1.0)
combined['pm10_so2_ratio'] = combined['PM10'] / (combined['SO2'] + 1.0)
combined['pm10_no2_ratio'] = combined['PM10'] / (combined['NO2'] + 1.0)
combined['so2_no2_ratio'] = combined['SO2'] / (combined['NO2'] + 1.0)
combined['co_no2_ratio'] = combined['CO'] / (combined['NO2'] + 1.0)
combined['o3_no2_ratio'] = combined['O3'] / (combined['NO2'] + 1.0)
combined['total_oxidant'] = combined['O3'] + combined['NO2']
combined['pm10_rh_inter'] = combined['PM10'] * (combined['RH'] / 100.0)
combined['pm10_temp_inter'] = combined['PM10'] * combined['TEMP']
combined['pm10_wspm_inter'] = combined['PM10'] / (combined['WSPM'] + 0.1)
combined['co_wspm_inter'] = combined['CO'] / (combined['WSPM'] + 0.1)

# Station Coordinates
station_coords = {
    'Dongsi': (39.929, 116.417), 'Tiantan': (39.886, 116.407),
    'Guanyuan': (39.929, 116.339), 'Wanshouxigong': (39.878, 116.352),
    'Aotizhongxin': (39.982, 116.397), 'Nongzhanguan': (39.937, 116.461),
    'Wanliu': (39.987, 116.287), 'Gucheng': (39.914, 116.184),
    'Shunyi': (40.127, 116.655), 'Changping': (40.217, 116.230),
    'Dingling': (40.292, 116.220), 'Huairou': (40.328, 116.628)
}
combined['lat'] = combined['station'].map(lambda s: station_coords[s][0])
combined['lon'] = combined['station'].map(lambda s: station_coords[s][1])
combined['dist_center'] = np.sqrt((combined['lat'] - 39.9042)**2 + (combined['lon'] - 116.4074)**2) * 111.0

# Synoptic Means & Anomalies
regional_cols = ['PM10', 'CO', 'NO2', 'SO2', 'TEMP', 'PRES', 'WSPM']
hourly_regional = combined.groupby('datetime')[regional_cols].transform('mean')
hourly_regional.columns = [f'regional_mean_{c}' for c in regional_cols]
combined = pd.concat([combined, hourly_regional], axis=1)

combined['pm10_anomaly'] = combined['PM10'] - combined['regional_mean_PM10']
combined['co_anomaly'] = combined['CO'] - combined['regional_mean_CO']
combined['no2_anomaly'] = combined['NO2'] - combined['regional_mean_NO2']

# Temporal Lags
tracking_cols = ['PM10', 'CO', 'NO2', 'SO2', 'TEMP', 'PRES', 'WSPM']
for col in tracking_cols:
    combined[f'{col}_lag1'] = combined.groupby('station')[col].shift(1)
    combined[f'{col}_lag2'] = combined.groupby('station')[col].shift(2)
    combined[f'{col}_diff1'] = combined[col] - combined[f'{col}_lag1']
    combined[f'{col}_diff3'] = combined[col] - combined.groupby('station')[col].shift(3)
    combined[f'{col}_roll_mean_3'] = combined.groupby('station')[col].transform(lambda x: x.rolling(3, min_periods=1).mean())
    combined[f'{col}_roll_mean_6'] = combined.groupby('station')[col].transform(lambda x: x.rolling(6, min_periods=1).mean())
    combined[f'{col}_roll_std_6'] = combined.groupby('station')[col].transform(lambda x: x.rolling(6, min_periods=1).std())
    combined[f'{col}_roll_mean_12'] = combined.groupby('station')[col].transform(lambda x: x.rolling(12, min_periods=1).mean())
    combined[f'{col}_roll_mean_24'] = combined.groupby('station')[col].transform(lambda x: x.rolling(24, min_periods=1).mean())
    combined[f'{col}_roll_max_24'] = combined.groupby('station')[col].transform(lambda x: x.rolling(24, min_periods=1).max())
    combined[f'{col}_roll_min_24'] = combined.groupby('station')[col].transform(lambda x: x.rolling(24, min_periods=1).min())

station_freq = combined['station'].value_counts(normalize=True).to_dict()
combined['station_freq'] = combined['station'].map(station_freq)
combined['station_code'] = combined['station'].astype('category').cat.codes
combined['wd_code'] = combined['wd'].astype('category').cat.codes

# Split back
train_df = combined[combined['is_test'] == 0].sort_values('orig_order').reset_index(drop=True)
test_df = combined[combined['is_test'] == 1].sort_values('orig_order').reset_index(drop=True)
train_df_clean = train_df.dropna(subset=['PM2.5']).copy()

excluded_cols = ['REF_NO', 'PM2.5', 'datetime', 'station', 'wd', 'year', 'orig_order', 'is_test', 'regional_mean_PM10', 'regional_mean_CO', 'regional_mean_NO2', 'regional_mean_SO2', 'regional_mean_TEMP', 'regional_mean_PRES', 'regional_mean_WSPM']
feature_cols = [c for c in train_df_clean.columns if c not in excluded_cols]
print(f"Total features created: {len(feature_cols)}")

tr_mask = train_df_clean['year'] < 2016
val_mask = train_df_clean['year'] == 2016

X_tr = train_df_clean.loc[tr_mask, feature_cols].copy()
y_tr = train_df_clean.loc[tr_mask, 'PM2.5'].copy()
X_val = train_df_clean.loc[val_mask, feature_cols].copy()
y_val = train_df_clean.loc[val_mask, 'PM2.5'].copy()

medians = X_tr.median()
X_tr_imp = X_tr.fillna(medians)
X_val_imp = X_val.fillna(medians)

models_eval = {}

# Ridge
ridge = Ridge(alpha=100.0)
ridge.fit(X_tr_imp, y_tr)
y_pred_ridge = np.clip(ridge.predict(X_val_imp), 0, None)
models_eval['Ridge'] = {'rmse': np.sqrt(mean_squared_error(y_val, y_pred_ridge)), 'mae': mean_absolute_error(y_val, y_pred_ridge), 'r2': r2_score(y_val, y_pred_ridge)}

# Random Forest (sampled)
rf = RandomForestRegressor(n_estimators=100, max_depth=16, max_features=0.5, random_state=42, n_jobs=4)
rf_sample_idx = np.random.RandomState(42).choice(len(X_tr_imp), size=min(80000, len(X_tr_imp)), replace=False)
rf.fit(X_tr_imp.iloc[rf_sample_idx], y_tr.iloc[rf_sample_idx])
y_pred_rf = np.clip(rf.predict(X_val_imp), 0, None)
models_eval['Random Forest'] = {'rmse': np.sqrt(mean_squared_error(y_val, y_pred_rf)), 'mae': mean_absolute_error(y_val, y_pred_rf), 'r2': r2_score(y_val, y_pred_rf)}

# LightGBM
lgb_tr_data = lgb.Dataset(X_tr, label=y_tr)
lgb_val_data = lgb.Dataset(X_val, label=y_val, reference=lgb_tr_data)
lgb_params = {
    'objective': 'regression', 'metric': 'rmse', 'learning_rate': 0.04, 'num_leaves': 127,
    'max_depth': -1, 'min_child_samples': 30, 'feature_fraction': 0.75, 'bagging_fraction': 0.8,
    'bagging_freq': 1, 'seed': 42, 'verbose': -1, 'n_jobs': 4
}
lgb_model = lgb.train(lgb_params, lgb_tr_data, num_boost_round=1500, valid_sets=[lgb_tr_data, lgb_val_data], callbacks=[lgb.early_stopping(50), lgb.log_evaluation(200)])
y_pred_lgb = np.clip(lgb_model.predict(X_val, num_iteration=lgb_model.best_iteration), 0, None)
models_eval['LightGBM'] = {'rmse': np.sqrt(mean_squared_error(y_val, y_pred_lgb)), 'mae': mean_absolute_error(y_val, y_pred_lgb), 'r2': r2_score(y_val, y_pred_lgb)}

# XGBoost
xgb_tr_data = xgb.DMatrix(X_tr, label=y_tr)
xgb_val_data = xgb.DMatrix(X_val, label=y_val)
xgb_params = {
    'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'tree_method': 'hist', 'learning_rate': 0.05,
    'max_depth': 8, 'colsample_bytree': 0.75, 'subsample': 0.8, 'seed': 42, 'nthread': 4
}
xgb_model = xgb.train(xgb_params, xgb_tr_data, num_boost_round=1200, evals=[(xgb_tr_data, 'train'), (xgb_val_data, 'val')], early_stopping_rounds=50, verbose_eval=200)
y_pred_xgb = np.clip(xgb_model.predict(xgb_val_data), 0, None)
models_eval['XGBoost'] = {'rmse': np.sqrt(mean_squared_error(y_val, y_pred_xgb)), 'mae': mean_absolute_error(y_val, y_pred_xgb), 'r2': r2_score(y_val, y_pred_xgb)}

# Blend
best_w = 0.55
y_pred_ensemble = best_w * y_pred_lgb + (1 - best_w) * y_pred_xgb
models_eval['Ensemble (LGB+XGB)'] = {
    'rmse': np.sqrt(mean_squared_error(y_val, y_pred_ensemble)),
    'mae': mean_absolute_error(y_val, y_pred_ensemble),
    'r2': r2_score(y_val, y_pred_ensemble)
}

summary_df = pd.DataFrame({
    'Model': list(models_eval.keys()),
    'RMSE': [models_eval[m]['rmse'] for m in models_eval],
    'MAE': [models_eval[m]['mae'] for m in models_eval],
    'R2': [models_eval[m]['r2'] for m in models_eval]
}).sort_values('RMSE').reset_index(drop=True)
print(summary_df)
summary_df.to_csv('model_plots/model_comparison.csv', index=False)

# Benchmark plot
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(summary_df['Model'], summary_df['RMSE'], color=['#1a9641', '#2ca25f', '#41b6c4', '#fdae61', '#d7191c'][::-1])
ax.set_xlabel('Validation RMSE (µg/m³)', fontsize=12)
for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.3, bar.get_y() + bar.get_height()/2, f'{w:.2f}', va='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('model_plots/01_model_rmse_comparison.png', bbox_inches='tight')
plt.close()

# Feature Importance
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': lgb_model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)
importance.to_csv('model_plots/feature_importance.csv', index=False)

fig, ax = plt.subplots(figsize=(12, 8))
top25 = importance.head(25)
sns.barplot(data=top25, x='importance', y='feature', hue='feature', palette='viridis', legend=False, ax=ax)
plt.tight_layout()
plt.savefig('model_plots/02_feature_importance.png', bbox_inches='tight')
plt.close()

# ========================================== #
# SECTION 3: RE-ALIGN AND GENERATE SUBMISSION #
# ========================================== #

print("\n=== RE-ALIGNING AND GENERATING EXACT-ORDER SUBMISSION ===")
X_full = train_df_clean[feature_cols]
y_full = train_df_clean['PM2.5']
X_test = test_df[feature_cols]

# Retrain on full dataset
ds_full = lgb.Dataset(X_full, label=y_full)
final_lgb = lgb.train(lgb_params, ds_full, num_boost_round=1100)

d_full = xgb.DMatrix(X_full, label=y_full)
final_xgb = xgb.train(xgb_params, d_full, num_boost_round=900)

d_test = xgb.DMatrix(X_test)
test_pred_lgb = np.clip(final_lgb.predict(X_test), 2.0, None)
test_pred_xgb = np.clip(final_xgb.predict(d_test), 2.0, None)
final_test_preds = 0.55 * test_pred_lgb + 0.45 * test_pred_xgb

sub_df = pd.DataFrame({
    'REF_NO': test_df['REF_NO'],
    'PM2.5': final_test_preds
})
sub_df.to_csv(os.path.join(base_dir, 'submission.csv'), index=False, header=False)
sub_df.to_csv(os.path.join(base_dir, 'submission_with_header.csv'), index=False, header=True)

print("Submission saved to /content/submission.csv")

# =========================================== #
# SECTION 4: LINEAR REGRESSION VISUALIZATION #
# =========================================== #

print("\n=== GENERATING LINEAR REGRESSION GRAPHS ===")
pollutant_cols = ['PM10', 'CO', 'NO2', 'SO2', 'O3', 'TEMP', 'PRES', 'DEWP', 'WSPM']
train_clean_lr = train.dropna(subset=['PM2.5'] + pollutant_cols).copy()

X_lr = train_clean_lr[pollutant_cols]
y_lr = train_clean_lr['PM2.5']

tr_mask_lr = (train_clean_lr['year'] < 2016).values
val_mask_lr = (train_clean_lr['year'] == 2016).values

X_tr_lr, y_tr_lr = X_lr.iloc[tr_mask_lr], y_lr.iloc[tr_mask_lr]
X_val_lr, y_val_lr = X_lr.iloc[val_mask_lr], y_lr.iloc[val_mask_lr]

scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr_lr)
X_val_scaled = scaler.transform(X_val_lr)

lr = LinearRegression()
lr.fit(X_tr_scaled, y_tr_lr)
y_pred_val_lr = lr.predict(X_val_scaled)

# Simulate gradient descent cost
def simulate_gradient_descent(X, y, lr_rate=0.08, n_iters=100):
    m, n = X.shape
    X_b = np.c_[np.ones((m, 1)), X]
    theta = np.zeros(n + 1)
    cost_history = []
    sub_idx = np.random.RandomState(42).choice(m, size=min(15000, m), replace=False)
    X_sub = X_b[sub_idx]
    y_sub = y.iloc[sub_idx].values
    m_sub = len(y_sub)
    for it in range(n_iters):
        preds = X_sub.dot(theta)
        error = preds - y_sub
        cost = (1.0 / (2.0 * m_sub)) * np.sum(error**2)
        cost_history.append(cost)
        gradient = (1.0 / m_sub) * X_sub.T.dot(error)
        theta = theta - lr_rate * gradient
    return cost_history

cost_history = simulate_gradient_descent(X_tr_scaled, y_tr_lr)

# Create 4-panel Linear Regression Fig
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
sample_idx = np.random.RandomState(42).choice(len(y_val_lr), size=min(25000, len(y_val_lr)), replace=False)
y_v_sub = y_val_lr.iloc[sample_idx]
y_p_sub = y_pred_val_lr[sample_idx]

hb = axes[0, 0].hexbin(y_v_sub, y_p_sub, gridsize=55, cmap='Blues', mincnt=1, bins='log')
axes[0, 0].plot([0, 600], [0, 600], 'r--', linewidth=2.5, label='Ideal Identity Line (Y = Ŷ)')
axes[0, 0].set_xlim(0, 550)
axes[0, 0].set_ylim(0, 550)
axes[0, 0].set_title('(A) Multivariate Linear Regression: Actual vs Predicted', fontsize=12, fontweight='bold')
fig.colorbar(hb, ax=axes[0, 0], label='log10(count)')

sample_plot = train_clean_lr.sample(min(15000, len(train_clean_lr)), random_state=42)
sns.regplot(data=sample_plot, x='PM10', y='PM2.5', scatter_kws={'alpha': 0.15, 'color': '#1f77b4', 's': 8},
            line_kws={'color': '#d95f02', 'linewidth': 2.5}, ax=axes[0, 1])
axes[0, 1].set_xlim(0, 600)
axes[0, 1].set_ylim(0, 600)
axes[0, 1].set_title('(B) Linear Regression Fit: PM2.5 vs PM10 ($r = +0.88$)', fontsize=12, fontweight='bold')

coef_df = pd.DataFrame({'Feature': pollutant_cols, 'Beta_Coefficient': lr.coef_}).sort_values('Beta_Coefficient', ascending=True)
colors = ['#d7191c' if c < 0 else '#2ca25f' for c in coef_df['Beta_Coefficient']]
bars = axes[1, 0].barh(coef_df['Feature'], coef_df['Beta_Coefficient'], color=colors)
axes[1, 0].set_title('(C) Learned Regression Coefficients (βⱼ Parameters)', fontsize=12, fontweight='bold')

axes[1, 1].plot(range(1, len(cost_history) + 1), cost_history, color='#7570b3', linewidth=2.5)
axes[1, 1].set_title('(D) Parameter Optimization via Gradient Descent', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(model_dir, 'linear_regression_comprehensive.png'), bbox_inches='tight')
plt.close()

print("Saved model_plots/linear_regression_comprehensive.png")
print(f"Finished entire execution successfully in {time.time() - start_time:.2f} seconds!")

--- STARTING EDA ---
All 8 EDA figures successfully saved to eda_plots/!

=== STARTING COMPREHENSIVE FEATURE ENGINEERING & MODEL BENCHMARK ===
Total features created: 130
Training until validation scores don't improve for 50 rounds
[200]	training's rmse: 13.7321	valid_1's rmse: 18.7642
[400]	training's rmse: 11.4297	valid_1's rmse: 18.3439
[600]	training's rmse: 10.164	valid_1's rmse: 18.2095
[800]	training's rmse: 9.26786	valid_1's rmse: 18.1148
[1000]	training's rmse: 8.57526	valid_1's rmse: 18.0483
Early stopping, best iteration is:
[1065]	training's rmse: 8.37889	valid_1's rmse: 18.034
[0]	train-rmse:76.53428	val-rmse:72.91491
[200]	train-rmse:12.50846	val-rmse:18.62105
[400]	train-rmse:10.26518	val-rmse:18.32382
[600]	train-rmse:8.93339	val-rmse:18.21856
[800]	train-rmse:7.94572	val-rmse:18.17955
[1000]	train-rmse:7.17713	val-rmse:18.15774
[1002]	train-rmse:7.17174	val-rmse:18.15757
                Model       RMSE        MAE        R2
0  Ensemble (LGB+XGB)  17.950231  10.210829  